In [8]:
from google.colab import files

uploaded = files.upload()

Saving patient_heart_rate.csv to patient_heart_rate (1).csv


In [13]:
import pandas as pd
import numpy as np
import io
from google.colab import files

# =========================================================================
# TỰ ĐỘNG TẠO FILE DỮ LIỆU CHUẨN LAB 3 THEO ĐÚNG SLIDE BÀI HỌC
# =========================================================================
print("🛠️ Hệ thống đang tự động khởi tạo file dữ liệu 'patient_heart_rate.csv' chuẩn theo slide...")

lab3_data = """1,Mickey Mouse,56,70kgs,72,69,71, ,72,68
2,Donald Duck,34,154lbs,91, ,89, , ,
3,  , , , , , , , ,
1,Mickey Mouse,56,70kgs,72,69,71, ,72,68
4,Minniê Mouse,45,50kgs, ,75, ,70,72,71
5,Goofy Dog,38,160lbs,85,82,88,78, ,79
"""

# Lưu file chuẩn vào bộ nhớ tạm của Colab
with open("patient_heart_rate.csv", "w", encoding="utf-8") as f:
    f.write(lab3_data)

# Cấu hình hiển thị bảng trên giao diện Colab
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# -------------------------------------------------------------------------
# 3. Vấn đề 1: Tiến hành tải dữ liệu vào chương trình Python và thêm header
# -------------------------------------------------------------------------
column_names = ["Id", "Name", "Age", "Weight", "m0006", "m0612", "m1218", "f0006", "f0612", "f1218"]
df = pd.read_csv("patient_heart_rate.csv", names=column_names)
print("\n--- Vấn đề 1: Thêm Header ---")
print(df.head())

# -------------------------------------------------------------------------
# 4. Vấn đề 2: Xử lý vấn đề một cột lưu hỗn hợp nhiều dữ liệu (Tách Name)
# -------------------------------------------------------------------------
# Đưa các khoảng trắng ẩn về dạng NaN để tránh lỗi khi tách
df['Name'] = df['Name'].replace(r'^\s*$', np.nan, regex=True)
df[['Firstname', 'Lastname']] = df['Name'].str.split(n=1, expand=True)
df = df.drop('Name', axis=1)
print("\n--- Vấn đề 2: Tách Name thành Firstname và Lastname ---")
print(df.head())

# -------------------------------------------------------------------------
# 5. Vấn đề 3: Cột Weight có vấn đề về không thống nhất các đơn vị đo lường
# -------------------------------------------------------------------------
df.reset_index(drop=True, inplace=True)
for i in range(0, len(df)):
    x = str(df.loc[i, 'Weight'])
    if "lbs" in x[-3:]:
        x = x[:-3]
        float_x = float(x)
        y = int(float_x / 2.2)
        y = str(y) + "kgs"
        df.loc[i, 'Weight'] = y

print("\n--- Vấn đề 3: Chuẩn hóa đơn vị Weight ---")
print(df.head())

# -------------------------------------------------------------------------
# 6. Vấn đề 4: Vấn đề xuất hiện dòng dữ liệu rỗng (NaN) -> Xóa bỏ
# -------------------------------------------------------------------------
df.dropna(how="all", inplace=True)

# -------------------------------------------------------------------------
# 7. Vấn đề 5: Có nhiều dòng dữ liệu bị trùng lặp thông tin hoàn toàn
# -------------------------------------------------------------------------
df.drop_duplicates(subset=['Firstname', 'Lastname', 'Age', 'Weight'], inplace=True)

# -------------------------------------------------------------------------
# 8. Vấn đề 6: Xuất hiện dữ liệu bị ảnh hưởng bởi lỗi non-ASCII
# -------------------------------------------------------------------------
df['Firstname'] = df['Firstname'].astype(str).replace({r'[^\x00-\x7F]+':''}, regex=True)
df['Lastname'] = df['Lastname'].astype(str).replace({r'[^\x00-\x7F]+':''}, regex=True)
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# -------------------------------------------------------------------------
# 10. Gom biến (Melt các cột Sex + Time Range thành một cột duy nhất)
# -------------------------------------------------------------------------
df = pd.melt(
    df,
    id_vars=['Id', 'Age', 'Weight', 'Firstname', 'Lastname'],
    value_name="PulseRate",
    var_name="sex_and_time"
).sort_values(['Id', 'Age', 'Weight', 'Firstname', 'Lastname'])

tmp_df = df["sex_and_time"].str.extract(r'(\D)(\d{2})(\d{2})', expand=True)
tmp_df.columns = ["Sex", "hours_lower", "hours_upper"]
tmp_df["Time"] = tmp_df["hours_lower"] + "-" + tmp_df["hours_upper"]

df = pd.concat([df, tmp_df], axis=1)
df = df.drop(['sex_and_time', 'hours_lower', 'hours_upper'], axis=1)

# -------------------------------------------------------------------------
# 9 & 11. Vấn đề 7: Khảo sát và xử lý dữ liệu thiếu trên biến huyết áp (PulseRate)
# -------------------------------------------------------------------------
df['PulseRate'] = pd.to_numeric(df['PulseRate'], errors='coerce')
df['PulseRate'] = df['PulseRate'].ffill().bfill()

# -------------------------------------------------------------------------
# 12. Rút gọn dữ liệu phù hợp, reindex lại dữ liệu và lưu file
# -------------------------------------------------------------------------
ordered_columns = ['Id', 'Firstname', 'Lastname', 'Age', 'Weight', 'Sex', 'Time', 'PulseRate']
df = df.reindex(columns=ordered_columns)

# Lưu trữ dữ liệu sạch và tự động tải về máy máy tính
df.to_csv('patient_heart_rate_clean.csv', index=False)
print("\n--- KẾT QUẢ CUỐI CÙNG SAU KHI LÀM SẠCH CHUẨN 12 BƯỚC SLIDE ---")
print(df.head(15))

print("\n📥 Đang tải file kết quả sạch 'patient_heart_rate_clean.csv' về máy tính của bạn...")
files.download('patient_heart_rate_clean.csv')

🛠️ Hệ thống đang tự động khởi tạo file dữ liệu 'patient_heart_rate.csv' chuẩn theo slide...

--- Vấn đề 1: Thêm Header ---
   Id          Name Age  Weight m0006 m0612 m1218 f0006 f0612  f1218
0   1  Mickey Mouse  56   70kgs    72    69    71          72   68.0
1   2   Donald Duck  34  154lbs    91          89                NaN
2   3                                                            NaN
3   1  Mickey Mouse  56   70kgs    72    69    71          72   68.0
4   4  Minniê Mouse  45   50kgs          75          70    72   71.0

--- Vấn đề 2: Tách Name thành Firstname và Lastname ---
   Id Age  Weight m0006 m0612 m1218 f0006 f0612  f1218 Firstname Lastname
0   1  56   70kgs    72    69    71          72   68.0    Mickey    Mouse
1   2  34  154lbs    91          89                NaN    Donald     Duck
2   3                                              NaN       NaN      NaN
3   1  56   70kgs    72    69    71          72   68.0    Mickey    Mouse
4   4  45   50kgs          75       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>